# Aula 4 — Enxergar além do verde

**Ideia central:** A planta estressada "apaga" no infravermelho antes de amarelar no visível. Índices espectrais são só razões entre bandas, mas transformam essa diferença num mapa de decisão.

**Ementa:** Assinatura espectral • Bandas e reflectância • NDVI, NDRE, SAVI • Zonas de manejo • Taxa variável

---

In [ ]:
# !pip install rasterio scikit-learn

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy import ndimage as ndi

## 1. Carregar bandas Sentinel-2 (nível L2A)

Baixe imagens do [Copernicus Open Access Hub](https://scihub.copernicus.eu/) ou do [Earth Explorer](https://earthexplorer.usgs.gov/).

In [ ]:
# Função para converter DN em reflectância
def refletancia(caminho):
    """Converte número digital (DN) do Sentinel-2 L2A em reflectância."""
    import rasterio
    with rasterio.open(caminho) as r:
        dn = r.read(1).astype("float32")
    # Desde 2022 (baseline 04.00) o L2A tem deslocamento de -1000
    return np.clip((dn - 1000) / 10000, 0, 1)

# Exemplo de uso (substitua pelos caminhos das suas bandas):
# red = refletancia("T21JYM_B04_10m.jp2")  # Vermelho
# nir = refletancia("T21JYM_B08_10m.jp2")  # Infravermelho próximo

## 2. Calcular NDVI

$$NDVI = \frac{NIR - Vermelho}{NIR + Vermelho}$$

In [ ]:
# Exemplo com dados sintéticos
np.random.seed(42)
nir = np.random.rand(100, 100) * 0.4 + 0.3  # Simula NIR
red = np.random.rand(100, 100) * 0.15 + 0.05  # Simula Vermelho

# Adiciona uma "mancha de estresse"
nir[30:50, 40:60] *= 0.6

# Cálculo do NDVI
ndvi = (nir - red) / (nir + red + 1e-6)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(red, cmap="Reds", vmin=0, vmax=0.3)
ax[0].set_title("Banda Vermelho")
ax[1].imshow(nir, cmap="Greys_r", vmin=0.2, vmax=0.7)
ax[1].set_title("Banda NIR")
im = ax[2].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=1)
ax[2].set_title("NDVI")
plt.colorbar(im, ax=ax[2])
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 3. Zonas de manejo com k-means

In [ ]:
# Suaviza para remover ruído
suave = ndi.median_filter(ndvi, size=5)

# Agrupa em 3 zonas
km = KMeans(n_clusters=3, n_init=10, random_state=0)
km.fit(suave.reshape(-1, 1))

# Ordena as zonas pelo NDVI médio (zona 0 = menor NDVI)
ordem = np.argsort(km.cluster_centers_.ravel())
zonas = np.argsort(ordem)[km.labels_].reshape(ndvi.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im0 = ax[0].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=1)
ax[0].set_title("NDVI")
plt.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(zonas, cmap="RdYlGn")
ax[1].set_title("Zonas de manejo (k=3)")
plt.colorbar(im1, ax=ax[1])
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

# Estatísticas por zona
print("Zona | NDVI médio | % da área")
print("-" * 35)
for z in range(3):
    mask = zonas == z
    print(f"  {z}  |   {ndvi[mask].mean():.3f}   |   {100 * mask.mean():.1f}%")

## 4. Série temporal no Google Earth Engine

Código para extrair série temporal de NDVI sem baixar imagens.

In [ ]:
# Requer autenticação no Google Earth Engine
# !pip install earthengine-api

# import ee
# ee.Authenticate()
# ee.Initialize(project="seu-projeto-no-gcp")

# ponto = ee.Geometry.Point([-54.09, -25.29]).buffer(300)  # Medianeira-PR
# col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
#          .filterBounds(ponto)
#          .filterDate("2025-10-01", "2026-03-31")
#          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)))

# def ndvi_medio(img):
#     v = img.normalizedDifference(["B8", "B4"]).reduceRegion(ee.Reducer.mean(), ponto, 10)
#     return ee.Feature(None, {"data": img.date().format("YYYY-MM-dd"), "ndvi": v.get("nd")})

# serie = ee.FeatureCollection(col.map(ndvi_medio)).getInfo()
# for f in serie["features"]:
#     print(f["properties"]["data"], round(f["properties"]["ndvi"] or 0, 3))

---

## Desafios

### Aquecimento
Monte uma tabela com os valores esperados de NDVI e NDRE para solo, palha, planta vigorosa e planta estressada.

### Mão na massa
No Earth Engine, extraia a série temporal de NDVI de um talhão na última safra de soja. Identifique emergência, fechamento e senescência.

### Investigação
Delineie zonas com k = 2 a 6 e calcule a redução de variância para cada k. A partir de qual k o ganho deixa de compensar?

In [ ]:
# Espaço para os desafios


---

## Leitura recomendada

- JENSEN, J. R. *Sensoriamento remoto do ambiente*. Parêntese, 2009.
- MOLIN, J. P.; AMARAL, L. R.; COLAÇO, A. F. *Agricultura de precisão*. Oficina de Textos, 2015.
- ROUSE, J. W. et al. Monitoring vegetation systems with ERTS. NASA, 1974.
- HUETE, A. R. A soil-adjusted vegetation index (SAVI). RSE, 1988.